In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 1. Загрузка и подготовка данных
# Для простоты предполагаем, что у нас есть user-item взаимодействия в виде матрицы

def load_mind_small_data(interactions_file, min_clicks=5):
    """
    Загрузка взаимодействий пользователей с новостями из файла.
    interactions_file: файл с взаимодействиями пользователей
    """
    interactions = pd.read_csv(interactions_file)
    user_click_counts = interactions['user_id'].value_counts()
    filtered_users = user_click_counts[user_click_counts >= min_clicks].index
    interactions = interactions[interactions['user_id'].isin(filtered_users)]
    
    # Создание словарей для маппинга
    user2id = {user: idx for idx, user in enumerate(interactions['user_id'].unique())}
    item2id = {item: idx for idx, item in enumerate(interactions['item_id'].unique())}
    
    num_users = len(user2id)
    num_items = len(item2id)
    
    # Создание матрицы взаимодействий
    interaction_matrix = np.zeros((num_users, num_items))
    for _, row in interactions.iterrows():
        uid = user2id[row['user_id']]
        iid = item2id[row['item_id']]
        interaction_matrix[uid, iid] = 1.0  # бинарная матрица
    
    return torch.FloatTensor(interaction_matrix), num_users, num_items

# 2. Определение VAE модели

class VAE(nn.Module):
    def __init__(self, num_items, hidden_dim=600, latent_dim=200):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(num_items, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_items),
            nn.Sigmoid()  # для бинарной реконструкции
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        hidden = self.encoder(x)
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decoder(z)
        return recon_x, mu, logvar

# 3. Dataset и Dataloader

class InteractionDataset(Dataset):
    def __init__(self, interaction_matrix):
        self.data = interaction_matrix
        
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        return self.data[idx]

# 4. Функция потерь VAE

def vae_loss(recon_x, x, mu, logvar):
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    # KL дивергенция
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

# 5. Обучение

def train_vae(model, dataloader, epochs=20, lr=1e-3, device='cpu'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    model.train()
    for epoch in range(epochs):
        train_loss = 0
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
            batch = batch.to(device)
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(batch)
            loss = vae_loss(recon_batch, batch, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()
        avg_loss = train_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")
    print("Training complete.")
    return model

# 6. Пример использования

if __name__ == "__main__":
    interaction_matrix, num_users, num_items, user2id, news2id = load_mind_small_behaviors("MINDsmall_train/behaviors.tsv")
    dataset = InteractionDataset(interaction_matrix)
    dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

    vae_model = VAE(num_items)
    trained_model = train_vae(vae_model, dataloader, epochs=20)

    # Пример: рекомендации для пользователя с индексом 0
    trained_model.eval()
    with torch.no_grad():
        user_vector = interaction_matrix[0].unsqueeze(0)
        recon_vector, _, _ = trained_model(user_vector)
        recommended_items = torch.topk(recon_vector[0], k=10).indices.numpy()
        print(f"Top-10 recommended items for user 0: {recommended_items}")



FileNotFoundError: [Errno 2] No such file or directory: 'interactions.csv'

In [1]:
import pandas as pd
import numpy as np
import torch

def load_mind_small_behaviors(behaviors_file, min_clicks=5):
    """
    Загружает interactions из behaviors.tsv
    и возвращает user-item interaction matrix.
    """
    behaviors = pd.read_csv(behaviors_file, sep='\t', header=None,
                            names=['ImpressionID', 'UserID', 'Time', 'History', 'Impressions'])
    
    # Соберем словарь пользователей
    user2id = {user: idx for idx, user in enumerate(behaviors['UserID'].unique())}
    # Соберем словарь новостей
    all_news = set()
    for history in behaviors['History'].dropna():
        all_news.update(history.strip().split(' '))
    for impressions in behaviors['Impressions']:
        for impression in impressions.strip().split(' '):
            news_id = impression.split('-')[0]
            all_news.add(news_id)
    news2id = {news: idx for idx, news in enumerate(all_news)}
    
    num_users = len(user2id)
    num_items = len(news2id)
    interaction_matrix = np.zeros((num_users, num_items))
    
    for _, row in behaviors.iterrows():
        user_id = row['UserID']
        impressions = row['Impressions'].strip().split(' ')
        for impression in impressions:
            news_id, clicked = impression.split('-')
            uid = user2id[user_id]
            iid = news2id[news_id]
            if clicked == '1':
                interaction_matrix[uid, iid] = 1.0  # бинарный клик
    
    return torch.FloatTensor(interaction_matrix), num_users, num_items, user2id, news2id


In [4]:
!pip install tqdm

     -------------------------------------- 78.5/78.5 kB 864.6 kB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
